# 🎯 YouTube Influencer Finder
### Find the Top 50 Influencers in Any Sector — Free, Open Source, No Subscription

---

This notebook uses the **YouTube Data API v3** (free, 10,000 units/day) to:

- Search for YouTube channels in a chosen sector
- Collect channel statistics: subscribers, total views, upload frequency
- Analyse recent video performance: views, likes, comments
- Calculate an **Influencer Strength Score** — a composite rating across reach, engagement, consistency, and authenticity
- Output a ranked table of the **Top 50 influencers** in your chosen niche

---

## 📋 Before You Start: Get Your Free API Key

You need a **YouTube Data API v3** key. This is free and takes about 5 minutes:

1. Go to [console.cloud.google.com](https://console.cloud.google.com/)
2. Create a new project (or select an existing one)
3. Navigate to **APIs & Services > Library**
4. Search for **"YouTube Data API v3"** and click **Enable**
5. Go to **APIs & Services > Credentials**
6. Click **Create Credentials > API Key**
7. Copy your key and paste it into **Cell 2** below

> **Free quota:** 10,000 units/day. This notebook uses roughly 300–600 units per run, so you can run it ~15–30 times per day comfortably.

---

## 📦 How the Strength Score Works

Each channel is rated out of **100** using five weighted components:

| Component | Weight | What It Measures |
|---|---|---|
| **Reach** | 25% | Subscriber count (log-scaled) |
| **Engagement Rate** | 30% | Avg (likes + comments) ÷ avg views |
| **View Authenticity** | 20% | Views-per-subscriber ratio — flags inflated follower counts |
| **Consistency** | 15% | Upload frequency over the past 90 days |
| **Momentum** | 10% | Recent view velocity vs. channel average |

> A high subscriber count with low engagement and poor views-per-subscriber is a strong proxy signal for fake or inactive followers — the same heuristic used by paid tools like Modash.

---
## Cell 1: Install & Import Dependencies

In [ ]:
# Install required libraries (only needed once)
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

install('google-api-python-client')
install('pandas')
install('numpy')
install('matplotlib')
install('seaborn')
install('tqdm')
install('ipywidgets')

print('✅ All dependencies installed.')

In [ ]:
# Core imports
import os
import time
import math
import warnings
from datetime import datetime, timezone, timedelta
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from tqdm.notebook import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print('✅ Imports successful.')

---
## Cell 2: Configuration — Enter Your API Key & Choose Your Sector

In [ ]:
# ============================================================
#  CONFIGURATION — Edit this cell before running
# ============================================================

# Paste your YouTube Data API v3 key here
API_KEY = 'YOUR_API_KEY_HERE'


# --- Sector Selection ---
# Available sectors and their associated search keywords.
# The tool will search YouTube using these terms to find relevant channels.

SECTORS = {
    '1':  ('Fitness & Health',             ['fitness', 'workout', 'gym', 'health tips', 'weight loss', 'yoga']),
    '2':  ('Finance & Investing',           ['personal finance', 'investing', 'stock market', 'financial independence', 'crypto investing']),
    '3':  ('Technology & Gadgets',          ['tech review', 'smartphone review', 'gadgets', 'technology news', 'unboxing tech']),
    '4':  ('Gaming',                        ['gaming', 'let\'s play', 'game review', 'esports', 'video games']),
    '5':  ('Beauty & Makeup',               ['makeup tutorial', 'beauty tips', 'skincare routine', 'cosmetics', 'beauty review']),
    '6':  ('Fashion & Style',               ['fashion tips', 'outfit ideas', 'style guide', 'fashion haul', 'clothing review']),
    '7':  ('Food & Cooking',                ['cooking', 'recipe', 'food review', 'baking', 'meal prep']),
    '8':  ('Travel & Adventure',            ['travel vlog', 'travel tips', 'adventure', 'travel guide', 'backpacking']),
    '9':  ('Business & Entrepreneurship',   ['entrepreneur', 'startup', 'business tips', 'side hustle', 'passive income']),
    '10': ('Education & Science',           ['science explained', 'education', 'learning', 'physics', 'biology']),
    '11': ('Music',                         ['music', 'singing', 'guitar tutorial', 'music production', 'covers']),
    '12': ('Sustainability & Environment',  ['sustainability', 'climate change', 'zero waste', 'eco-friendly', 'renewable energy']),
    '13': ('Parenting & Family',            ['parenting tips', 'family vlog', 'pregnancy', 'kids activities', 'homeschooling']),
    '14': ('Mental Health & Wellbeing',     ['mental health', 'anxiety', 'mindfulness', 'self improvement', 'therapy']),
    '15': ('Sports',                        ['football', 'basketball', 'sport highlights', 'athletics', 'sport analysis']),
    '16': ('Art & Design',                  ['digital art', 'drawing tutorial', 'graphic design', 'illustration', 'art tips']),
    '17': ('Cars & Automotive',             ['car review', 'automotive', 'car modification', 'electric vehicle', 'road test']),
    '18': ('DIY & Home Improvement',        ['DIY', 'home improvement', 'woodworking', 'home decor', 'renovation']),
    '19': ('Pets & Animals',               ['dog training', 'cat videos', 'pet care', 'animal rescue', 'exotic pets']),
    '20': ('Politics & Current Affairs',   ['politics', 'news analysis', 'current affairs', 'political commentary', 'world news']),
}

# Print menu
print('\n🎯 Available Sectors:\n')
for k, (name, _) in SECTORS.items():
    print(f'  [{k:>2}]  {name}')

print('\n' + '─'*45)
print('Set CHOSEN_SECTOR_ID to your chosen number below.')

In [ ]:
# ============================================================
#  Set your chosen sector ID from the list above
# ============================================================

CHOSEN_SECTOR_ID = '1'   # <-- Change this number

# ---- Advanced settings (optional) ----
TOP_N             = 50    # How many influencers to return
MAX_RESULTS_PER_KEYWORD = 10  # Channels fetched per search keyword (max 50)
RECENT_VIDEOS_TO_ANALYSE = 10  # Recent videos used for engagement scoring
MIN_SUBSCRIBERS   = 1000  # Filter out micro-accounts below this threshold
# ----------------------------------------

sector_name, search_keywords = SECTORS[CHOSEN_SECTOR_ID]
print(f'✅ Selected sector: {sector_name}')
print(f'   Search keywords: {', '.join(search_keywords)}')

---
## Cell 3: YouTube API Client & Helper Functions

In [ ]:
# Initialise YouTube API client

if API_KEY == 'YOUR_API_KEY_HERE':
    raise ValueError('⛔  Please replace YOUR_API_KEY_HERE with your actual API key in Cell 2.')

youtube = build('youtube', 'v3', developerKey=API_KEY)
print('✅ YouTube API client ready.')


# ── Helper: search for channels by keyword ──────────────────────────────────
def search_channels(keyword: str, max_results: int = 10) -> list[str]:
    """Return a list of channel IDs matching a search keyword."""
    channel_ids = []
    try:
        response = youtube.search().list(
            q=keyword,
            part='snippet',
            type='channel',
            maxResults=min(max_results, 50),
            relevanceLanguage='en',
            order='relevance'
        ).execute()
        for item in response.get('items', []):
            cid = item['snippet'].get('channelId') or item['id'].get('channelId')
            if cid:
                channel_ids.append(cid)
    except HttpError as e:
        print(f'  ⚠️  API error on keyword "{keyword}": {e}')
    return channel_ids


# ── Helper: get channel statistics ──────────────────────────────────────────
def get_channel_stats(channel_ids: list[str]) -> dict:
    """Fetch statistics and metadata for a list of channel IDs (batched in 50s)."""
    results = {}
    for i in range(0, len(channel_ids), 50):
        batch = channel_ids[i:i+50]
        try:
            response = youtube.channels().list(
                part='snippet,statistics,contentDetails,brandingSettings',
                id=','.join(batch)
            ).execute()
            for item in response.get('items', []):
                cid   = item['id']
                stats = item.get('statistics', {})
                snip  = item.get('snippet', {})
                results[cid] = {
                    'channel_id':        cid,
                    'channel_name':      snip.get('title', 'Unknown'),
                    'description':       snip.get('description', '')[:200],
                    'country':           snip.get('country', 'N/A'),
                    'created_at':        snip.get('publishedAt', ''),
                    'thumbnail':         snip.get('thumbnails', {}).get('default', {}).get('url', ''),
                    'subscribers':       int(stats.get('subscriberCount', 0)),
                    'total_views':       int(stats.get('viewCount', 0)),
                    'video_count':       int(stats.get('videoCount', 0)),
                    'hidden_subscribers': stats.get('hiddenSubscriberCount', False),
                    'uploads_playlist':  item.get('contentDetails', {}).get('relatedPlaylists', {}).get('uploads', ''),
                }
        except HttpError as e:
            print(f'  ⚠️  Channel stats error: {e}')
    return results


# ── Helper: get recent video IDs from uploads playlist ──────────────────────
def get_recent_video_ids(uploads_playlist_id: str, max_videos: int = 10) -> list[str]:
    """Return the IDs of the most recent uploads from a channel's uploads playlist."""
    video_ids = []
    try:
        response = youtube.playlistItems().list(
            part='contentDetails',
            playlistId=uploads_playlist_id,
            maxResults=min(max_videos, 50)
        ).execute()
        for item in response.get('items', []):
            vid = item['contentDetails'].get('videoId')
            if vid:
                video_ids.append(vid)
    except HttpError:
        pass
    return video_ids


# ── Helper: get video statistics ────────────────────────────────────────────
def get_video_stats(video_ids: list[str]) -> list[dict]:
    """Fetch statistics for a list of video IDs."""
    stats_list = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        try:
            response = youtube.videos().list(
                part='statistics,snippet',
                id=','.join(batch)
            ).execute()
            for item in response.get('items', []):
                s = item.get('statistics', {})
                published = item.get('snippet', {}).get('publishedAt', '')
                stats_list.append({
                    'video_id':     item['id'],
                    'views':        int(s.get('viewCount', 0)),
                    'likes':        int(s.get('likeCount', 0)),
                    'comments':     int(s.get('commentCount', 0)),
                    'published_at': published,
                })
        except HttpError:
            pass
    return stats_list


print('✅ Helper functions defined.')

---
## Cell 4: Data Collection — Search & Fetch Channel Data

In [ ]:
print(f'🔍 Searching YouTube for channels in: {sector_name}\n')
print(f'   Using {len(search_keywords)} keywords, up to {MAX_RESULTS_PER_KEYWORD} results each.\n')

# --- Step 1: Collect unique channel IDs across all keywords ---
all_channel_ids = set()
keyword_bar = tqdm(search_keywords, desc='Searching keywords')

for keyword in keyword_bar:
    keyword_bar.set_postfix({'keyword': keyword[:30]})
    ids = search_channels(keyword, max_results=MAX_RESULTS_PER_KEYWORD)
    all_channel_ids.update(ids)
    time.sleep(0.2)  # Polite rate limiting

print(f'\n📊 Found {len(all_channel_ids)} unique channels across all keywords.')
print('   Fetching channel statistics...')

In [ ]:
# --- Step 2: Fetch channel statistics ---
channel_data = get_channel_stats(list(all_channel_ids))

# Filter: remove channels with hidden subscriber counts or below threshold
filtered = {
    cid: data for cid, data in channel_data.items()
    if not data['hidden_subscribers']
    and data['subscribers'] >= MIN_SUBSCRIBERS
    and data['uploads_playlist'] != ''
}

print(f'✅ Retrieved stats for {len(channel_data)} channels.')
print(f'   After filtering (min {MIN_SUBSCRIBERS:,} subscribers, public counts): {len(filtered)} channels remain.')

In [ ]:
# --- Step 3: Fetch recent video stats for each channel ---
print(f'\n🎬 Fetching recent video data ({RECENT_VIDEOS_TO_ANALYSE} videos per channel)...\n')

now = datetime.now(timezone.utc)
ninety_days_ago = now - timedelta(days=90)

channel_video_metrics = {}

for cid, data in tqdm(filtered.items(), desc='Analysing channels'):
    playlist_id = data['uploads_playlist']
    if not playlist_id:
        continue

    video_ids = get_recent_video_ids(playlist_id, max_videos=RECENT_VIDEOS_TO_ANALYSE)
    if not video_ids:
        continue

    video_stats = get_video_stats(video_ids)
    if not video_stats:
        continue

    # Count uploads in last 90 days
    recent_uploads = 0
    for vs in video_stats:
        if vs['published_at']:
            try:
                pub = datetime.fromisoformat(vs['published_at'].replace('Z', '+00:00'))
                if pub >= ninety_days_ago:
                    recent_uploads += 1
            except ValueError:
                pass

    views_list    = [v['views']    for v in video_stats if v['views']    > 0]
    likes_list    = [v['likes']    for v in video_stats]
    comments_list = [v['comments'] for v in video_stats]

    avg_views    = np.mean(views_list)    if views_list    else 0
    avg_likes    = np.mean(likes_list)    if likes_list    else 0
    avg_comments = np.mean(comments_list) if comments_list else 0
    std_views    = np.std(views_list)     if len(views_list) > 1 else 0

    # Most recent video views vs channel average (momentum)
    momentum = (views_list[0] / avg_views) if avg_views > 0 and views_list else 1.0

    channel_video_metrics[cid] = {
        'avg_views':      avg_views,
        'avg_likes':      avg_likes,
        'avg_comments':   avg_comments,
        'std_views':      std_views,
        'recent_uploads': recent_uploads,
        'momentum':       momentum,
        'videos_analysed': len(video_stats),
    }
    time.sleep(0.1)

print(f'\n✅ Video metrics collected for {len(channel_video_metrics)} channels.')

---
## Cell 5: Scoring Engine — Calculate Influencer Strength Score

In [ ]:
def min_max_scale(values: np.ndarray, log_transform: bool = False) -> np.ndarray:
    """Normalise values to [0, 1]. Optionally apply log transform for skewed distributions."""
    if log_transform:
        values = np.log1p(values)
    min_v, max_v = values.min(), values.max()
    if max_v == min_v:
        return np.zeros_like(values, dtype=float)
    return (values - min_v) / (max_v - min_v)


def clamp(value: float, min_val: float = 0.0, max_val: float = 1.0) -> float:
    return max(min_val, min(max_val, value))


# ── Assemble the master DataFrame ───────────────────────────────────────────
rows = []
for cid, ch in filtered.items():
    if cid not in channel_video_metrics:
        continue
    vm = channel_video_metrics[cid]
    rows.append({**ch, **vm})

df = pd.DataFrame(rows)
print(f'Master dataset: {len(df)} channels')

if df.empty:
    raise ValueError('No data collected — check your API key and try again.')


# ── Component 1: Reach Score (subscriber count, log-scaled) ─────────────────
df['reach_score'] = min_max_scale(df['subscribers'].values.astype(float), log_transform=True)


# ── Component 2: Engagement Rate ────────────────────────────────────────────
# (avg likes + avg comments) / avg views — capped at 1.0 to handle edge cases
df['raw_engagement_rate'] = df.apply(
    lambda r: clamp((r['avg_likes'] + r['avg_comments']) / r['avg_views'])
    if r['avg_views'] > 0 else 0.0, axis=1
)
df['engagement_score'] = min_max_scale(df['raw_engagement_rate'].values.astype(float))


# ── Component 3: Authenticity Score (views-per-subscriber ratio) ─────────────
# Higher views per subscriber = more genuine audience.
# A channel with 1M subs but only 5k avg views is suspicious.
df['views_per_sub'] = df.apply(
    lambda r: r['avg_views'] / r['subscribers'] if r['subscribers'] > 0 else 0.0, axis=1
)
# Log scale this too — distribution is very skewed
df['authenticity_score'] = min_max_scale(df['views_per_sub'].values.astype(float), log_transform=True)


# ── Component 4: Consistency Score (uploads in last 90 days) ─────────────────
# Score of 1.0 = 12+ uploads in 90 days (~1/week). Capped there.
df['consistency_score'] = df['recent_uploads'].apply(lambda x: clamp(x / 12.0))


# ── Component 5: Momentum Score (recent vs average views) ────────────────────
# Capped: 2x average view count = full score. Below 0.5x = low score.
df['momentum_score'] = df['momentum'].apply(lambda x: clamp((x - 0.5) / 1.5))


# ── Composite Strength Score (weighted sum × 100) ────────────────────────────
WEIGHTS = {
    'reach_score':       0.25,
    'engagement_score':  0.30,
    'authenticity_score':0.20,
    'consistency_score': 0.15,
    'momentum_score':    0.10,
}

df['strength_score'] = sum(
    df[col] * weight for col, weight in WEIGHTS.items()
) * 100

df['strength_score'] = df['strength_score'].round(1)


# ── Sort and select Top N ─────────────────────────────────────────────────────
df_top = df.sort_values('strength_score', ascending=False).head(TOP_N).reset_index(drop=True)
df_top.index += 1  # Rank starts at 1

print(f'\n✅ Scoring complete. Top {TOP_N} influencers ranked.')

---
## Cell 6: Results — Top 50 Influencer Table

In [ ]:
# ── Helper: format large numbers for readability ─────────────────────────────
def fmt(n):
    if n >= 1_000_000:
        return f'{n/1_000_000:.1f}M'
    if n >= 1_000:
        return f'{n/1_000:.1f}K'
    return str(int(n))


# ── Build display table ───────────────────────────────────────────────────────
display_cols = [
    'channel_name', 'country', 'subscribers', 'avg_views',
    'raw_engagement_rate', 'views_per_sub', 'recent_uploads',
    'strength_score'
]

df_display = df_top[display_cols].copy()
df_display.columns = [
    'Channel', 'Country', 'Subscribers', 'Avg Views',
    'Engagement Rate', 'Views/Sub', 'Uploads (90d)',
    'Strength Score'
]

df_display['Subscribers']    = df_display['Subscribers'].apply(fmt)
df_display['Avg Views']      = df_display['Avg Views'].apply(lambda x: fmt(int(x)))
df_display['Engagement Rate']= df_display['Engagement Rate'].apply(lambda x: f'{x*100:.2f}%')
df_display['Views/Sub']      = df_display['Views/Sub'].apply(lambda x: f'{x:.3f}')
df_display['Strength Score'] = df_display['Strength Score'].apply(lambda x: f'{x:.1f} / 100')

print(f'\n🏆  TOP {TOP_N} YOUTUBE INFLUENCERS: {sector_name.upper()}\n')
print('='*120)
display(df_display.style
    .background_gradient(subset=['Strength Score'], cmap='YlGn')
    .set_properties(**{'text-align': 'left', 'font-size': '13px'})
    .set_table_styles([{'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left')]}])
)

---
## Cell 7: Visualisations

In [ ]:
# ── Plot 1: Top 20 by Strength Score (horizontal bar chart) ──────────────────
fig, ax = plt.subplots(figsize=(12, 8))

top20 = df_top.head(20).iloc[::-1]  # Reverse for bottom-to-top ordering

bars = ax.barh(
    top20['channel_name'], top20['strength_score'],
    color=plt.cm.YlOrRd(top20['strength_score'] / 100),
    edgecolor='white', linewidth=0.5
)

for bar, score in zip(bars, top20['strength_score']):
    ax.text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
        f'{score:.1f}', va='center', ha='left', fontsize=10, fontweight='bold'
    )

ax.set_xlabel('Strength Score (out of 100)', fontsize=12)
ax.set_title(f'Top 20 YouTube Influencers: {sector_name}', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, 110)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('top20_strength_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as top20_strength_score.png')

In [ ]:
# ── Plot 2: Engagement Rate vs Subscribers (scatter — authenticity map) ───────
fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(
    df_top['subscribers'],
    df_top['raw_engagement_rate'] * 100,
    c=df_top['strength_score'],
    cmap='plasma',
    s=100,
    alpha=0.8,
    edgecolors='white',
    linewidths=0.5
)

plt.colorbar(scatter, ax=ax, label='Strength Score')

# Label top 10
for _, row in df_top.head(10).iterrows():
    ax.annotate(
        row['channel_name'][:20],
        (row['subscribers'], row['raw_engagement_rate'] * 100),
        fontsize=7.5, ha='left', va='bottom',
        xytext=(5, 3), textcoords='offset points'
    )

ax.set_xscale('log')
ax.set_xlabel('Subscribers (log scale)', fontsize=12)
ax.set_ylabel('Engagement Rate (%)', fontsize=12)
ax.set_title(f'Engagement vs Reach — {sector_name}\n(Brighter = Higher Strength Score)', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: fmt(int(x))))
plt.tight_layout()
plt.savefig('engagement_vs_reach.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as engagement_vs_reach.png')

In [ ]:
# ── Plot 3: Score breakdown radar for the top 5 channels ─────────────────────
score_cols   = ['reach_score', 'engagement_score', 'authenticity_score', 'consistency_score', 'momentum_score']
score_labels = ['Reach', 'Engagement', 'Authenticity', 'Consistency', 'Momentum']

top5 = df_top.head(5)
N    = len(score_labels)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), subplot_kw=dict(polar=True))

colours = plt.cm.tab10.colors

for i, (_, row) in enumerate(top5.iterrows()):
    ax = axes[i]
    values = [row[c] for c in score_cols]
    values += values[:1]

    ax.fill(angles, values, alpha=0.25, color=colours[i])
    ax.plot(angles, values, linewidth=2, color=colours[i])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(score_labels, fontsize=8)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels([])
    ax.set_title(row['channel_name'][:18], fontsize=9, fontweight='bold', pad=12)
    ax.set_ylim(0, 1)

fig.suptitle(f'Score Breakdown — Top 5 Channels in {sector_name}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('radar_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as radar_top5.png')

---
## Cell 8: Export Results

In [ ]:
# Export the full scored dataset to CSV
timestamp     = datetime.now().strftime('%Y%m%d_%H%M')
sector_slug   = sector_name.lower().replace(' ', '_').replace('&', 'and')
csv_filename  = f'influencers_{sector_slug}_{timestamp}.csv'

export_cols = [
    'channel_id', 'channel_name', 'country', 'subscribers', 'total_views', 'video_count',
    'avg_views', 'avg_likes', 'avg_comments', 'raw_engagement_rate', 'views_per_sub',
    'recent_uploads', 'momentum',
    'reach_score', 'engagement_score', 'authenticity_score', 'consistency_score', 'momentum_score',
    'strength_score'
]

df_top[export_cols].to_csv(csv_filename, index=True, index_label='rank')

print(f'✅ Results exported to: {csv_filename}')
print(f'   {len(df_top)} channels saved.')

---
## Cell 9: Channel Deep-Dive (Optional)

Enter any channel name from the results to see a detailed breakdown.

In [ ]:
# ── Set a channel name to inspect (must match a result in df_top) ─────────────
CHANNEL_TO_INSPECT = df_top.iloc[0]['channel_name']  # Defaults to rank #1
# Change to e.g. CHANNEL_TO_INSPECT = 'MrBeast' to inspect a specific channel

match = df_top[df_top['channel_name'].str.lower() == CHANNEL_TO_INSPECT.lower()]

if match.empty:
    print(f'Channel "{CHANNEL_TO_INSPECT}" not found in top {TOP_N} results.')
else:
    row = match.iloc[0]
    rank = match.index[0]

    print(f'\n{'='*55}')
    print(f'  📺  {row["channel_name"]}')
    print(f'{'='*55}')
    print(f'  Rank:             #{rank}')
    print(f'  Country:          {row["country"]}')
    print(f'  Subscribers:      {fmt(row["subscribers"])}')
    print(f'  Total Views:      {fmt(row["total_views"])}')
    print(f'  Videos Published: {int(row["video_count"]):,}')
    print(f'  Channel URL:      https://www.youtube.com/channel/{row["channel_id"]}')
    print()
    print(f'  --- Performance (last {RECENT_VIDEOS_TO_ANALYSE} videos) ---')
    print(f'  Avg Views:        {fmt(int(row["avg_views"]))}')
    print(f'  Avg Likes:        {fmt(int(row["avg_likes"]))}')
    print(f'  Avg Comments:     {fmt(int(row["avg_comments"]))}')
    print(f'  Engagement Rate:  {row["raw_engagement_rate"]*100:.2f}%')
    print(f'  Views / Sub:      {row["views_per_sub"]:.3f}')
    print(f'  Uploads (90d):    {int(row["recent_uploads"])}')
    print()
    print(f'  --- Score Breakdown ---')
    print(f'  Reach:            {row["reach_score"]*100:.1f}/100   (weight: 25%)')
    print(f'  Engagement:       {row["engagement_score"]*100:.1f}/100   (weight: 30%)')
    print(f'  Authenticity:     {row["authenticity_score"]*100:.1f}/100   (weight: 20%)')
    print(f'  Consistency:      {row["consistency_score"]*100:.1f}/100   (weight: 15%)')
    print(f'  Momentum:         {row["momentum_score"]*100:.1f}/100   (weight: 10%)')
    print(f'  ──────────────────────────────')
    print(f'  STRENGTH SCORE:   {row["strength_score"]:.1f} / 100  🏆')
    print(f'{'='*55}')

---

## 📌 Notes & Limitations

- **API Quota:** The free YouTube Data API v3 provides 10,000 units/day. Each run of this notebook costs roughly 300–700 units depending on the number of channels found. If you hit your quota, wait until midnight Pacific Time (when it resets) or create a second Google Cloud project.

- **Fake follower detection:** This notebook uses *views-per-subscriber ratio* and *engagement rate* as proxy signals for audience authenticity. This is the same heuristic used by many paid tools. It is not a definitive audit, but low scores on these metrics are a reliable red flag worth investigating.

- **Search coverage:** YouTube's search API returns results by relevance, not exhaustively. A small number of niche creators may not appear. Increasing `MAX_RESULTS_PER_KEYWORD` (up to 50) or adding more keywords will improve coverage at the cost of more API quota.

- **Hidden subscriber counts:** Some channels hide their subscriber count. These are excluded automatically since they cannot be scored fairly.

- **Instagram / TikTok:** These platforms are deliberately excluded due to API restrictions. See the project README for a discussion of future extension possibilities.

---
*Built with the YouTube Data API v3. Not affiliated with Google or YouTube.*